In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from streameucliddes.analysis import euclid_des_preparation as eprep
from streameucliddes.plotting.plotting import plot_CMD, plot_2D_density, plot_2D_density_from_hp
from streameucliddes.injection import generate, inject
from streameucliddes.plotting import plot_style

from streamobs.match_filter import build_match_filter, is_in_match_filter

import astropy.coordinates as coord
import astropy.units as u
import gala.coordinates as gc

# surveys

In [ ]:
from streamobs.surveys import Survey

In [ ]:
des = Survey.load("des", release="yr6")
euclid = Survey.load("euclid", release="q1")

# Injection

In [ ]:
STREAM_CONFIG = {
    "stream_length": 7.0,  # degrees
    "stream_width": 0.2,  # degrees
    #"N": 10000,  # Number of stars
    "surface_brightness": 33.,
    "distance_modulus":{
    "center": {"type": "Constant", "value": 16.8},
    "spread": {"type": "Constant", "value": 0.0},
},
    "isochrone": {
        "name": "Marigo2017",
        "age": 12.0,  # Gyr
        "z": 0.0006,  # metallicity
        "surveys": ["des", "euclid"],
    }
}

end1 = coord.SkyCoord(ra=67 * u.deg, dec=-47 * u.deg, frame="icrs")
end2 = coord.SkyCoord(ra=55 * u.deg, dec=-49 * u.deg, frame="icrs")
gc_frame = gc.GreatCircleICRSFrame.from_endpoints(end1, end2)

In [ ]:
bands = {"euclid_q1": ["VIS", "Y", "J", "H"],"des_yr6": ["g", "r", "i"],}

injected_catalog, metadata = inject.inject_stream(
        stream_config=STREAM_CONFIG, seed=42, 
        bands=bands, 
        gc_frame=gc_frame, 
        detection_mag_cut=[], # if we want to avoid applying snr cuts
    )

In [ ]:
flag_obs_euclid = injected_catalog['euclid_q1_flag_observed']
flag_obs_des = injected_catalog['des_yr6_flag_observed']
flag_obs_cross = flag_obs_euclid & flag_obs_des
observed_stream_euclid = injected_catalog[flag_obs_euclid]
observed_stream_des = injected_catalog[flag_obs_des]
observed_stream_both = injected_catalog[flag_obs_cross]
observed_stream_any = injected_catalog[flag_obs_euclid | flag_obs_des]
print(f"Injected {len(injected_catalog)} stars into the catalog.")
print(f"Observed {len(observed_stream_euclid)} stars in the Euclid band.")
print(f"Observed {len(observed_stream_des)} stars in the DES band.")
print(f"Observed {len(observed_stream_both)} stars in both surveys.")
print(f"Observed {len(observed_stream_any)} stars in any survey.")

In [ ]:
footprint_des = des.coverage.copy()
footprint_euclid = euclid.coverage.copy()
footprint_des[footprint_des < 1] = np.nan
footprint_euclid[footprint_euclid < 1] = np.nan
footprint_des_euclid = footprint_des * footprint_euclid

fig, ax = plt.subplots(figsize=(8, 6))
fig, ax = plot_2D_density_from_hp(footprint_des_euclid, fig=fig, ax=ax, add_colorbar=False)
ax.scatter(injected_catalog["ra"], injected_catalog["dec"], s=1, color="red", label="Injected Stream")


In [ ]:
print(f"Injected {len(injected_catalog)} stars into the catalog.")
print("Columns in the injected catalog:", injected_catalog.columns.tolist())
print("Metadata:", metadata.keys())

In [ ]:
stars_background = eprep.get_eucliddes_stars()

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(10, 18))
axes = axes.flatten()

ylim = (16, 26.5)
xlim = (-1.0, 2.5)

# g,r 
plot_CMD(stars_background, fig=fig, ax=axes[0], mag_cols=["des_yr6_g_obs", "des_yr6_r_obs"],
         xlim=xlim, ylim=ylim, xlabel="$g - r$", ylabel="$g$")
axes[0].scatter(injected_catalog["des_g_true"] - injected_catalog["des_r_true"],
             injected_catalog["des_g_true"], color="red", s=10, label="True Stream")

# I-H
plot_CMD(stars_background, fig=fig, ax=axes[1], mag_cols=["euclid_q1_VIS_obs", "euclid_q1_H_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$VIS - H_E$", ylabel=r"$VIS$")
axes[1].scatter(injected_catalog["euclid_VIS_true"] - injected_catalog["euclid_H_true"],
             injected_catalog["euclid_VIS_true"], color="red", s=10, label="True Stream")

# I-Y
plot_CMD(stars_background, fig=fig, ax=axes[2], mag_cols=["euclid_q1_VIS_obs", "euclid_q1_Y_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$VIS - Y_E$", ylabel=r"$VIS$")
axes[2].scatter(injected_catalog["euclid_VIS_true"] - injected_catalog["euclid_Y_true"],
             injected_catalog["euclid_VIS_true"], color="red", s=10, label="True Stream")

# I-J
plot_CMD(stars_background, fig=fig, ax=axes[3], mag_cols=["euclid_q1_VIS_obs", "euclid_q1_J_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$VIS - J_E$", ylabel=r"$VIS$")
axes[3].scatter(injected_catalog["euclid_VIS_true"] - injected_catalog["euclid_J_true"],
             injected_catalog["euclid_VIS_true"], color="red", s=10, label="True Stream")

# J-H
plot_CMD(stars_background, fig=fig, ax=axes[4], mag_cols=["euclid_q1_J_obs", "euclid_q1_H_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$J_E - H_E$", ylabel=r"$J_E$")
axes[4].scatter(injected_catalog["euclid_J_true"] - injected_catalog["euclid_H_true"],
             injected_catalog["euclid_J_true"], color="red", s=10, label="True Stream")

# g-I
plot_CMD(stars_background, fig=fig, ax=axes[5], mag_cols=["des_yr6_g_obs", "euclid_q1_VIS_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$g - VIS$", ylabel="$g$")
axes[5].scatter(injected_catalog["des_g_true"] - injected_catalog["euclid_VIS_true"],
             injected_catalog["des_g_true"], color="red", s=10, label="True Stream")

# g-H
plot_CMD(stars_background, fig=fig, ax=axes[6], mag_cols=["des_yr6_g_obs", "euclid_q1_H_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$g - H_E$", ylabel="$g$")
axes[6].scatter(injected_catalog["des_g_true"] - injected_catalog["euclid_H_true"],
             injected_catalog["des_g_true"], color="red", s=10, label="True Stream")

# g - i
plot_CMD(stars_background, fig=fig, ax=axes[7], mag_cols=["des_yr6_g_obs", "des_yr6_i_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$g - i$", ylabel="$g$")
axes[7].scatter(injected_catalog["des_g_true"] - injected_catalog["des_i_true"],
             injected_catalog["des_g_true"], color="red", s=10, label="True Stream")

# r - H
plot_CMD(stars_background, fig=fig, ax=axes[8], mag_cols=["des_yr6_r_obs", "euclid_q1_H_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$r - H_E$", ylabel="$r$")
axes[8].scatter(injected_catalog["des_r_true"] - injected_catalog["euclid_H_true"],
             injected_catalog["des_r_true"], color="red", s=10, label="True Stream")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

# g,r 
plot_CMD(observed_stream_des, fig=fig, ax=axes[0], mag_cols=["des_yr6_g_obs", "des_yr6_r_obs"],
         xlim=(-0.5, 2.0), ylim=(16, 26), xlabel="$g - r$", ylabel="$g$")
axes[0].scatter(injected_catalog["des_g_true"] - injected_catalog["des_r_true"],
             injected_catalog["des_g_true"], color="red", s=10, label="True Stream")

# I-H
plot_CMD(observed_stream_euclid, fig=fig, ax=axes[1], mag_cols=["euclid_q1_VIS_obs", "euclid_q1_H_obs"],
         xlim=(-1.0, 2.5), ylim=(16, 26), xlabel=r"$VIS - H_E$", ylabel=r"$VIS$")
axes[1].scatter(injected_catalog["euclid_VIS_true"] - injected_catalog["euclid_H_true"],
             injected_catalog["euclid_VIS_true"], color="red", s=10, label="True Stream")

# g-I
plot_CMD(observed_stream_both, fig=fig, ax=axes[2], mag_cols=["des_yr6_g_obs", "euclid_q1_VIS_obs"],
         xlim=(-0.5, 3.0), ylim=(16, 26), xlabel=r"$g - VIS$", ylabel="$g$")
axes[2].scatter(injected_catalog["des_g_true"] - injected_catalog["euclid_VIS_true"],
             injected_catalog["des_g_true"], color="red", s=10, label="True Stream")

# g-H
plot_CMD(observed_stream_both, fig=fig, ax=axes[3], mag_cols=["des_yr6_g_obs", "euclid_q1_H_obs"],
         xlim=(-0.5, 4.0), ylim=(16, 26), xlabel=r"$g - H_E$", ylabel="$g$")
axes[3].scatter(injected_catalog["des_g_true"] - injected_catalog["euclid_H_true"],
             injected_catalog["des_g_true"], color="red", s=10, label="True Stream")


# Matched filter selection

In [ ]:
observed_stream_both['is_stream'] = True
stars_background['is_stream'] = False
total_stars = pd.concat([observed_stream_both, stars_background])

In [ ]:
stream_coord = coord.SkyCoord(
    ra=total_stars["ra"].to_numpy() * u.deg,
    dec=total_stars["dec"].to_numpy() * u.deg,
    frame="icrs",
).transform_to(gc_frame)

total_stars["phi1"] = stream_coord.phi1.deg
total_stars["phi2"] = stream_coord.phi2.deg

In [ ]:
cuts = {"des_yr6_g_err":(None,0.2), "des_yr6_r_err":(None,0.2), 
        "euclid_q1_VIS_err":(None,0.2), "euclid_q1_H_err":(None,0.2),
        "phi2":(-10.,10.)}
mask_cuts = eprep.get_cut_mask(total_stars, cuts)

In [ ]:
flag_stream = total_stars['is_stream']

In [ ]:
DISTANCE_MODULUS = STREAM_CONFIG["distance_modulus"]["center"]["value"]
AGE = STREAM_CONFIG["isochrone"]["age"]
METALLICITY = STREAM_CONFIG["isochrone"]["z"]

# Creating match filter shape in color-magnitude space
mf_des_gr = build_match_filter(
    distance_modulus=DISTANCE_MODULUS,
    age=AGE,
    metallicity=METALLICITY,
    band_1="g",
    band_2="r",
    survey_1="des",
    survey_2="des",
)
# Selecting stars in the stream and background catalogs that are inside the match filter polygon
flag_mf_des_gr = is_in_match_filter(
    total_stars["des_yr6_g_obs"],
    total_stars["des_yr6_r_obs"],
    polygon_vertices=mf_des_gr,
)

print(
    f"Stream selected by match filter: {flag_mf_des_gr.sum()} / {len(total_stars)} "
    f"({flag_mf_des_gr.mean()*100:.1f}%)"
)


mf_euclid_VISH = build_match_filter(
    distance_modulus=DISTANCE_MODULUS,
    age=AGE,
    metallicity=METALLICITY,
    band_1="VIS",
    band_2="H",
    survey_1="euclid",
    survey_2="euclid",
)

flag_mf_euclid_VISH = is_in_match_filter(
    total_stars["euclid_q1_VIS_obs"],
    total_stars["euclid_q1_H_obs"],
    polygon_vertices=mf_euclid_VISH,
)
print(
    f"Stream selected by match filter: {flag_mf_euclid_VISH.sum()} / {len(total_stars)} "
    f"({flag_mf_euclid_VISH.mean()*100:.1f}%)"
)

mf_cross_gH = build_match_filter(
    distance_modulus=DISTANCE_MODULUS,
    age=AGE,
    metallicity=METALLICITY,
    band_1="g",
    band_2="H",
    survey_1="des",
    survey_2="euclid",
    red_cap_value = 2.
)
flag_cross_gH = is_in_match_filter(
    total_stars["des_yr6_g_obs"],
    total_stars["euclid_q1_H_obs"],
    polygon_vertices=mf_cross_gH,
)
print(
    f"Stream selected by match filter: {flag_cross_gH.sum()} / {len(total_stars)} "
    f"({flag_cross_gH.mean()*100:.1f}%)"
)

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(10, 18))
axes = axes.flatten()

ylim = (16, 26.5)
xlim = (-1.0, 2.5)

# g,r 
plot_CMD(stars_background, fig=fig, ax=axes[0], mag_cols=["des_yr6_g_obs", "des_yr6_r_obs"],
         xlim=xlim, ylim=ylim, xlabel="$g - r$", ylabel="$g$")
axes[0].scatter(injected_catalog["des_g_true"] - injected_catalog["des_r_true"],
             injected_catalog["des_g_true"], color="red", s=10, label="True Stream")
mf = build_match_filter(
    distance_modulus=DISTANCE_MODULUS,
    age=AGE,
    metallicity=METALLICITY,
    band_1="g",
    band_2="r",
    survey_1="des",
    survey_2="des",
)
axes[0].plot(mf[:, 0], mf[:, 1], color="black", lw=2, label="Match Filter")

# I-H
plot_CMD(stars_background, fig=fig, ax=axes[1], mag_cols=["euclid_q1_VIS_obs", "euclid_q1_H_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$VIS - H_E$", ylabel=r"$VIS$")
axes[1].scatter(injected_catalog["euclid_VIS_true"] - injected_catalog["euclid_H_true"],
             injected_catalog["euclid_VIS_true"], color="red", s=10, label="True Stream")
mf = build_match_filter(
    distance_modulus=DISTANCE_MODULUS,
    age=AGE,
    metallicity=METALLICITY,
    band_1="VIS",
    band_2="H",
    survey_1="euclid",
    survey_2="euclid",
)
axes[1].plot(mf[:, 0], mf[:, 1], color="black", lw=2, label="Match Filter")

# I-Y
plot_CMD(stars_background, fig=fig, ax=axes[2], mag_cols=["euclid_q1_VIS_obs", "euclid_q1_Y_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$VIS - Y_E$", ylabel=r"$VIS$")
axes[2].scatter(injected_catalog["euclid_VIS_true"] - injected_catalog["euclid_Y_true"],
             injected_catalog["euclid_VIS_true"], color="red", s=10, label="True Stream")
mf = build_match_filter(
    distance_modulus=DISTANCE_MODULUS,
    age=AGE,
    metallicity=METALLICITY,
    band_1="VIS",
    band_2="Y",
    survey_1="euclid",
    survey_2="euclid",
)
axes[2].plot(mf[:, 0], mf[:, 1], color="black", lw=2, label="Match Filter")

# I-J
plot_CMD(stars_background, fig=fig, ax=axes[3], mag_cols=["euclid_q1_VIS_obs", "euclid_q1_J_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$VIS - J_E$", ylabel=r"$VIS$")
axes[3].scatter(injected_catalog["euclid_VIS_true"] - injected_catalog["euclid_J_true"],
             injected_catalog["euclid_VIS_true"], color="red", s=10, label="True Stream")
mf = build_match_filter(
    distance_modulus=DISTANCE_MODULUS,
    age=AGE,
    metallicity=METALLICITY,
    band_1="VIS",
    band_2="J",
    survey_1="euclid",
    survey_2="euclid",
)
axes[3].plot(mf[:, 0], mf[:, 1], color="black", lw=2, label="Match Filter")

# J-H
plot_CMD(stars_background, fig=fig, ax=axes[4], mag_cols=["euclid_q1_J_obs", "euclid_q1_H_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$J_E - H_E$", ylabel=r"$J_E$")
axes[4].scatter(injected_catalog["euclid_J_true"] - injected_catalog["euclid_H_true"],
             injected_catalog["euclid_J_true"], color="red", s=10, label="True Stream")
mf = build_match_filter(
    distance_modulus=DISTANCE_MODULUS,
    age=AGE,
    metallicity=METALLICITY,
    band_1="J",
    band_2="H",
    survey_1="euclid",
    survey_2="euclid",
)
axes[4].plot(mf[:, 0], mf[:, 1], color="black", lw=2, label="Match Filter")


# g-I
plot_CMD(stars_background, fig=fig, ax=axes[5], mag_cols=["des_yr6_g_obs", "euclid_q1_VIS_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$g - VIS$", ylabel="$g$")
axes[5].scatter(injected_catalog["des_g_true"] - injected_catalog["euclid_VIS_true"],
             injected_catalog["des_g_true"], color="red", s=10, label="True Stream")
mf = build_match_filter(
    distance_modulus=DISTANCE_MODULUS,
    age=AGE,
    metallicity=METALLICITY,
    band_1="g",
    band_2="VIS",
    survey_1="des",
    survey_2="euclid",
)
axes[5].plot(mf[:, 0], mf[:, 1], color="black", lw=2, label="Match Filter")


# g-H
plot_CMD(stars_background, fig=fig, ax=axes[6], mag_cols=["des_yr6_g_obs", "euclid_q1_H_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$g - H_E$", ylabel="$g$")
axes[6].scatter(injected_catalog["des_g_true"] - injected_catalog["euclid_H_true"],
             injected_catalog["des_g_true"], color="red", s=10, label="True Stream")
mf = build_match_filter(
    distance_modulus=DISTANCE_MODULUS,
    age=AGE,
    metallicity=METALLICITY,
    band_1="g",
    band_2="H",
    survey_1="des",
    survey_2="euclid",
    red_cap_value = 2.
)
axes[6].plot(mf[:, 0], mf[:, 1], color="black", lw=2, label="Match Filter")

# g - i
plot_CMD(stars_background, fig=fig, ax=axes[7], mag_cols=["des_yr6_g_obs", "des_yr6_i_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$g - i$", ylabel="$g$")
axes[7].scatter(injected_catalog["des_g_true"] - injected_catalog["des_i_true"],
             injected_catalog["des_g_true"], color="red", s=10, label="True Stream")
mf = build_match_filter(
    distance_modulus=DISTANCE_MODULUS,
    age=AGE,
    metallicity=METALLICITY,
    band_1="g",
    band_2="i",
    survey_1="des",
    survey_2="des",
)
axes[7].plot(mf[:, 0], mf[:, 1], color="black", lw=2, label="Match Filter")


# r - H
plot_CMD(stars_background, fig=fig, ax=axes[8], mag_cols=["des_yr6_r_obs", "euclid_q1_H_obs"],
         xlim=xlim, ylim=ylim, xlabel=r"$r - H_E$", ylabel="$r$")
axes[8].scatter(injected_catalog["des_r_true"] - injected_catalog["euclid_H_true"],
             injected_catalog["des_r_true"], color="red", s=10, label="True Stream")
mf = build_match_filter(
    distance_modulus=DISTANCE_MODULUS,
    age=AGE,
    metallicity=METALLICITY,
    band_1="r",
    band_2="H",
    survey_1="des",
    survey_2="euclid",
)
axes[8].plot(mf[:, 0], mf[:, 1], color="black", lw=2, label="Match Filter")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

# g,r 
plot_CMD(total_stars[mask_cuts], fig=fig, ax=axes[0], mag_cols=["des_yr6_g_obs", "des_yr6_r_obs"],
         xlim=(-0.5, 2.0), ylim=(16, 26), xlabel="$g - r$", ylabel="$g$")
axes[0].scatter(injected_catalog["des_g_true"] - injected_catalog["des_r_true"],
             injected_catalog["des_g_true"], color="red", s=10, label="True Stream")
# plot match filter polygon
axes[0].plot(mf_des_gr[:, 0], mf_des_gr[:, 1], color="black", lw=2, label="Match Filter")

# I-H
plot_CMD(total_stars[mask_cuts], fig=fig, ax=axes[1], mag_cols=["euclid_q1_VIS_obs", "euclid_q1_H_obs"],
         xlim=(-1.0, 2.5), ylim=(16, 26), xlabel=r"$VIS - H_E$", ylabel=r"$VIS$")
axes[1].scatter(injected_catalog["euclid_VIS_true"] - injected_catalog["euclid_H_true"],
             injected_catalog["euclid_VIS_true"], color="red", s=10, label="True Stream")
axes[1].plot(mf_euclid_VISH[:, 0], mf_euclid_VISH[:, 1], color="black", lw=2, label="Match Filter")

# g-H
plot_CMD(total_stars[mask_cuts], fig=fig, ax=axes[3], mag_cols=["des_yr6_g_obs", "euclid_q1_H_obs"],
         xlim=(-0.5, 4.0), ylim=(16, 26), xlabel=r"$g - H_E$", ylabel="$g$")
axes[3].scatter(injected_catalog["des_g_true"] - injected_catalog["euclid_H_true"],
             injected_catalog["des_g_true"], color="red", s=10, label="True Stream")
axes[3].plot(mf_cross_gH[:, 0], mf_cross_gH[:, 1], color="black", lw=2, label="Match Filter")


In [ ]:
plot_2D_density(
    total_stars[mask_cuts&flag_mf_des_gr],
    columns=("ra", "dec"),
    nside=512,
    nest=False,
    fig=None,
    ax=None,
    lon_range=None,
    lat_range=None,
    title=None,
    cmap="Greys",
    vmin=None,
    vmax=None,
    add_colorbar=True,
)

In [ ]:
plot_2D_density(
    total_stars[mask_cuts&flag_mf_euclid_VISH],
    columns=("ra", "dec"),
    nside=512,
    nest=False,
    fig=None,
    ax=None,
    lon_range=None,
    lat_range=None,
    title=None,
    cmap="Greys",
    vmin=None,
    vmax=None,
    add_colorbar=True,
)

In [ ]:
plot_2D_density(
    total_stars[mask_cuts&flag_cross_gH],
    columns=("ra", "dec"),
    nside=512,
    nest=False,
    fig=None,
    ax=None,
    lon_range=None,
    lat_range=None,
    title=None,
    cmap="Greys",
    vmin=None,
    vmax=None,
    add_colorbar=True,
)

In [ ]:
cuts_snr = {
        "phi2":(-1.4*STREAM_CONFIG['stream_width'],1.4*STREAM_CONFIG['stream_width']),}
mask_cuts_snr = eprep.get_cut_mask(total_stars, cuts_snr) & mask_cuts

In [ ]:
def get_SNR(data, mask):
    # Compute SNR for the selected data
    signal = len(data[mask & flag_stream])
    noise = len(data[mask & ~flag_stream])
    if noise == 0:
        return np.inf  # Avoid division by zero
    return signal / np.sqrt(noise)

In [ ]:
print("Matched filter DES (g-r) + SNR cuts: ", get_SNR(total_stars, mask_cuts_snr&flag_mf_des_gr))
print("Matched filter Euclid (VIS-H) + SNR cuts: ", get_SNR(total_stars, mask_cuts_snr&flag_mf_euclid_VISH))
print("Matched filter cross (g-H) + SNR cuts: ", get_SNR(total_stars, mask_cuts_snr&flag_cross_gH))